# Building Agents with LangGraph

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

In the earlier notebooks we built agents *from scratch* — handwritten
tool-use loops, manual message management, ad-hoc state. That was on purpose:
seeing the loop is the only way to understand what frameworks are doing
underneath.

But once a workflow gets bigger (branching, retries, human approval,
checkpointing, streaming, multiple agents), the boilerplate becomes the bug.
**LangGraph** is the most widely adopted framework for organising agent logic
as an explicit *graph* — nodes are computations, edges are control flow,
state is shared and persisted.

## What you'll learn

1. The core LangGraph primitives: `StateGraph`, `MessagesState`, `ToolNode`,
   `tools_condition`, `MemorySaver`.
2. How to build a single-agent ReAct-style loop in ~30 lines.
3. How `MemorySaver` gives you free **conversation persistence** keyed by
   `thread_id` — what would have taken pages of code in our hand-rolled agents.
4. How to add a **conditional branch** (e.g. early-exit on low confidence).
5. When LangGraph helps and when it gets in the way.

We'll re-implement the microscopy particle-analysis agent from
`04_agent_microscopy.ipynb` so you can compare the two side by side.

---
## Setup

In [ ]:
# !pip install langgraph langchain-anthropic numpy scipy

In [ ]:
import os, getpass, numpy as np
from typing import Annotated
from typing_extensions import TypedDict

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

MODEL = "claude-sonnet-4-6" 

---
## Step 1: Define tools

LangChain's `@tool` decorator turns a typed Python function into a tool the
LLM can call. The docstring becomes the tool description, type hints become
the schema — same idea as the Anthropic SDK, just wrapped one level higher
to be model-agnostic.

In [ ]:
from scipy import ndimage

# A synthetic microscopy image, shared by all tool calls (same idea as
# notebook 04's SESSION dict — kept module-level for clarity).
def _make_image(seed=0, n=45, size=512, mean_r=14, std_r=4, gap=4):
    rng = np.random.default_rng(seed)
    img = np.zeros((size, size), dtype=np.float32)
    yy, xx = np.mgrid[:size, :size]
    placed = []
    while len(placed) < n:
        r = float(np.clip(rng.normal(mean_r, std_r), 6, 28))
        cx, cy = rng.uniform(r+2, size-r-2, size=2)
        if any((cx-px)**2 + (cy-py)**2 < (r+pr+gap)**2 for px,py,pr in placed):
            continue
        placed.append((cx, cy, r))
    for cx, cy, r in placed:
        mask = (xx-cx)**2 + (yy-cy)**2 <= r**2
        img[mask] = 0.9
    img += 0.04 * rng.standard_normal(img.shape)
    return np.clip(img, 0, 1)

SESSION = {"image": _make_image(), "binary": None, "labels": None, "n": 0}

@tool
def threshold_image(method: str = "otsu") -> dict:
    """Binarize the loaded microscopy image into foreground/background.

    Args:
        method: 'otsu' for automatic thresholding (only option supported here).
    """
    img = SESSION["image"]
    hist, edges = np.histogram(img, bins=256, range=(0, 1))
    p = hist / hist.sum()
    omega = np.cumsum(p)
    mu = np.cumsum(p * (edges[:-1] + 0.5*(edges[1]-edges[0])))
    sigma_b = (mu[-1]*omega - mu)**2 / (omega*(1-omega) + 1e-12)
    thr = float(edges[:-1][np.argmax(sigma_b)])
    SESSION["binary"] = (img > thr).astype(np.uint8)
    return {"threshold": thr, "foreground_fraction": float(SESSION["binary"].mean())}

@tool
def segment_particles(min_area: int = 30) -> dict:
    """Connected-component labelling on the binarized image."""
    if SESSION["binary"] is None:
        return {"error": "Call threshold_image first."}
    labels, n = ndimage.label(SESSION["binary"])
    sizes = ndimage.sum(SESSION["binary"], labels, range(1, n+1))
    keep = np.where(sizes >= min_area)[0] + 1
    relabel = np.zeros(n+1, dtype=int)
    relabel[keep] = np.arange(1, len(keep)+1)
    SESSION["labels"], SESSION["n"] = relabel[labels], len(keep)
    return {"n_particles": int(SESSION["n"])}

@tool
def measure_particles() -> dict:
    """Mean/median/std of equivalent radius (in pixels) over labelled particles."""
    labels, n = SESSION["labels"], SESSION["n"]
    if labels is None or n == 0:
        return {"error": "Call segment_particles first."}
    areas = ndimage.sum(np.ones_like(labels), labels, range(1, n+1))
    radii = np.sqrt(areas / np.pi)
    return {"n_particles": int(n),
            "mean_radius_px":   float(radii.mean()),
            "median_radius_px": float(np.median(radii)),
            "std_radius_px":    float(radii.std())}

TOOLS = [threshold_image, segment_particles, measure_particles]
print([t.name for t in TOOLS])

---
## Step 2: Define the graph state

State is a TypedDict. The `messages` field uses `add_messages` as its
*reducer* — that means new messages are *appended* rather than overwriting.
LangGraph already provides `MessagesState` as a shortcut, but writing it
out once shows what's happening.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

---
## Step 3: Build the model node

A *node* is just a function `state -> state-update`. Our model node:
- reads the conversation so far,
- calls Claude (with tools bound),
- returns the new assistant message to be appended.

In [ ]:
llm = ChatAnthropic(model=MODEL, temperature=0).bind_tools(TOOLS)

def chatbot(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

---
## Step 4: Wire the graph

Now the structural part. The graph is:

```
   START -> chatbot ─┐
                     │ tools_condition
              tools <┘  (loops back to chatbot)
                ↓
            chatbot
                ↓
              END (no more tool calls)
```

`tools_condition` is a prebuilt function: if the last message contains a
tool call it returns `"tools"`, otherwise it returns `END`.

In [ ]:
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(TOOLS))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")

memory = MemorySaver()                       # in-process checkpointer
graph = builder.compile(checkpointer=memory)
print("Graph compiled with nodes:", list(graph.nodes))

**Try this** in your editor: `print(graph.get_graph().draw_ascii())`
prints an ASCII diagram of the compiled graph. (In Jupyter,
`graph.get_graph().draw_mermaid_png()` renders a Mermaid diagram.)

---
## Step 5: Run the agent

We pass a `thread_id` in the config — this is the key to memory. Every
invocation with the same `thread_id` continues the same conversation;
different `thread_id` = fresh conversation.

In [ ]:
SYSTEM = ("You are a microscopy data analyst. A microscopy image is already "
          "loaded. Use the tools to threshold, segment, and measure the particles. "
          "Report a concise quantitative summary at the end.")

cfg = {"configurable": {"thread_id": "session-001"}}

initial = {"messages": [
    SystemMessage(content=SYSTEM),
    HumanMessage(content=(
        "How many particles are in the loaded image, and what is the "
        "mean equivalent radius? Also describe the spread."
    )),
]}

final_state = graph.invoke(initial, cfg)

print("=== Final answer ===\n")
print(final_state["messages"][-1].content)

Internally LangGraph just executed the same loop you wrote by hand in
04_agent_microscopy.ipynb — but the structure is now declarative, the state
is automatically checkpointed, and you got streaming/persistence for free.

---
## Step 6: Persistence — pick up where you left off

Because we passed a `MemorySaver` checkpointer and a `thread_id`, the entire
conversation is now stored. We can ask a *follow-up* question and the model
sees the original context.

In [ ]:
follow_up = {"messages": [HumanMessage(
    content="Re-segment with min_area=200 and tell me if the count changes much."
)]}

state_2 = graph.invoke(follow_up, cfg)
print(state_2["messages"][-1].content)

Notice we *didn't* re-state which image we're working on — the assistant
remembers from the prior turn because LangGraph reloaded the conversation
state from the checkpointer. Switch the `thread_id` to something else and you'd
start fresh.

---
## Step 7: Streaming intermediate steps

For UIs (and for debugging) you usually want to *see* each step instead of
waiting for the final state. `graph.stream(...)` yields one chunk per node
execution.

In [ ]:
cfg2 = {"configurable": {"thread_id": "session-002"}}
init = {"messages": [
    SystemMessage(content=SYSTEM),
    HumanMessage(content="Threshold the image, segment with min_area=50, and report the count."),
]}

for chunk in graph.stream(init, cfg2):
    for node_name, node_state in chunk.items():
        msg = node_state["messages"][-1]
        kind = type(msg).__name__
        # The last message is either an AIMessage (with tool_calls or text) or a ToolMessage
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print(f"[{node_name}/{kind}] tool calls: {[tc['name'] for tc in msg.tool_calls]}")
        else:
            content = msg.content if isinstance(msg.content, str) else str(msg.content)
            print(f"[{node_name}/{kind}] {content[:160]}")

---
## Step 8: Adding a custom branch

The whole point of using a graph is that you can do things a flat loop can't.
Suppose we want an **early-exit safety check**: if the user's question looks
out of scope, route to a "polite refusal" node instead of running the agent.

In [ ]:
from langchain_core.messages import AIMessage

def safety_check(state: State) -> str:
    """Conditional edge: returns the name of the next node."""
    last_user = next((m for m in reversed(state["messages"])
                      if isinstance(m, HumanMessage)), None)
    if last_user and "weather" in last_user.content.lower():
        return "refuse"
    return "chatbot"

def refuse(state: State) -> dict:
    return {"messages": [AIMessage(
        content="I'm a microscopy analysis agent — I can't answer weather questions."
    )]}

builder2 = StateGraph(State)
builder2.add_node("chatbot", chatbot)
builder2.add_node("tools", ToolNode(TOOLS))
builder2.add_node("refuse", refuse)
builder2.add_conditional_edges(START, safety_check, {"chatbot": "chatbot", "refuse": "refuse"})
builder2.add_conditional_edges("chatbot", tools_condition)
builder2.add_edge("tools", "chatbot")
builder2.add_edge("refuse", END)
graph2 = builder2.compile()

# Try both paths
for q in ["What's the weather in Honolulu?", "How many particles are in the loaded image?"]:
    out = graph2.invoke({"messages": [SystemMessage(content=SYSTEM), HumanMessage(content=q)]})
    print(f"\nQ: {q}\nA: {out['messages'][-1].content[:160]}")

That's the LangGraph mental model in one cell: **nodes are functions**,
**edges are routing rules**, and *state* threads through both. Once you have
that, adding human-in-the-loop, retries, parallel branches, sub-graphs, etc.
is mostly more of the same primitives.

---
## When to use LangGraph (and when not to)

**Reach for LangGraph when:**
- The workflow has branches, retries, or human approval steps.
- You need conversation persistence across requests (e.g. a chat UI).
- You want to swap LLM providers without rewriting the loop.
- You'll deploy to LangSmith / LangGraph Cloud for tracing & observability.

**Stay with the SDK directly when:**
- The task is a one-shot "send prompt, get answer" with maybe one tool.
- You need every byte of token control (LangGraph adds some message wrapping).
- You're shipping a library and don't want to take on the dependency.

Either way, the *concepts* you learned in the from-scratch notebooks
(`messages`, `tool_use`, `tool_result`, the loop) are exactly what's running
under the hood. The framework is a convenience, not a magic trick.

## Further reading

- LangGraph docs: <https://langchain-ai.github.io/langgraph/>
- ChatAnthropic integration: <https://docs.langchain.com/oss/python/integrations/chat/anthropic>
- LangSmith for tracing & evals: <https://smith.langchain.com>